# Family History data analysis (Snowflake, read-only)

One row = one family history entry (a condition reported for one relative). **Read-only**: `SELECT` / `DESCRIBE` only.

**Cell format:** analysis cells are **SQL cells**, so each result shows Table / Chart / Pivot and the **download** button.

**Grain:** `FamilyHistoryId` should be unique. One patient can have many entries (several conditions, several relatives).

**The three views you asked for**
- Section 7 `condition_counts` — every unique `Condition` with its occurrences.
- Section 8 `snomed_condition_pairs` — one row per SNOMED code + condition name, with occurrences.
- Section 8 `snomed_name_variations` and `snomed_same_code_many_names` — the same code written with different condition names.

## 1. Active session

In [ ]:
import pandas as pd

from snowflake.snowpark.context import get_active_session

session = get_active_session()
session

## 2. Config (change names ONLY here)

SQL cells can only read **plain string** variables, so this cell exposes flat names (`T`, `C_COND`, `C_SNOMED`, ...).

`Condition` and `Status` are ordinary words in SQL, so they are quoted. Keep `QUOTE_COLUMNS = True` unless `DESCRIBE` shows plain uppercase names.

In [ ]:
DATABASE_NAME = "ATTR"
SCHEMA_NAME = "PUBLIC"
TABLE_NAME = "FAMILY_HISTORY"   # try FAMILYHISTORY, FAMILY_HX, FAM_HISTORY

QUOTE_DATABASE = False
QUOTE_SCHEMA = False
QUOTE_TABLE = False
QUOTE_COLUMNS = True

COL = {
    "family_history_id": "FamilyHistoryId",
    "encounter_id": "EncounterId/VisitId",
    "patient_id": "Member/PatientId",
    "snomed": "SNOMED",
    "condition": "Condition",
    "status": "Status",
    "family_member": "FamilyMember",
    "date": "Date",
}


def sf_ident(name, quoted):
    if quoted:
        return '"' + str(name).replace('"', '""') + '"'
    return str(name)


def col(key):
    return sf_ident(COL[key], QUOTE_COLUMNS)


# Flat strings for SQL cells
DB = sf_ident(DATABASE_NAME, QUOTE_DATABASE)
T = ".".join(
    [
        DB,
        sf_ident(SCHEMA_NAME, QUOTE_SCHEMA),
        sf_ident(TABLE_NAME, QUOTE_TABLE),
    ]
)

C_ID = col("family_history_id")
C_ENC = col("encounter_id")
C_PT = col("patient_id")
C_SNOMED = col("snomed")
C_COND = col("condition")
C_STATUS = col("status")
C_MEMBER = col("family_member")
C_DATE = col("date")

for name, value in [
    ("DB", DB),
    ("T", T),
    ("C_ID", C_ID),
    ("C_ENC", C_ENC),
    ("C_PT", C_PT),
    ("C_SNOMED", C_SNOMED),
    ("C_COND", C_COND),
    ("C_STATUS", C_STATUS),
    ("C_MEMBER", C_MEMBER),
    ("C_DATE", C_DATE),
]:
    print(f"{name} = {value}")

## 3. Find the table (only if the name or schema is wrong)

In [ ]:
SELECT
    CURRENT_ROLE() AS ROLE,
    CURRENT_WAREHOUSE() AS WAREHOUSE,
    CURRENT_DATABASE() AS DATABASE,
    CURRENT_SCHEMA() AS SCHEMA;

In [ ]:
SELECT
    TABLE_CATALOG,
    TABLE_SCHEMA,
    TABLE_NAME,
    ROW_COUNT,
    BYTES
FROM {{DB}}.INFORMATION_SCHEMA.TABLES
WHERE TABLE_TYPE = 'BASE TABLE'
  AND (
        UPPER(TABLE_NAME) LIKE '%FAMILY%'
     OR UPPER(TABLE_NAME) LIKE '%HISTORY%'
     OR UPPER(TABLE_NAME) LIKE '%FAM%'
  )
ORDER BY TABLE_SCHEMA, TABLE_NAME;

## 4. Table shape and first 10 rows

If a later cell fails with **invalid identifier**, copy the exact names from this `DESCRIBE` result into `COL`.

In [ ]:
DESCRIBE TABLE {{T}};

In [ ]:
SELECT *
FROM {{T}}
LIMIT 10;

## 5. Volume and uniqueness

Expected: unique `FamilyHistoryId` close to row count. Fewer unique patients means people report several conditions or several relatives.

In [ ]:
SELECT
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_ID}}) AS UNIQUE_FAMILY_HISTORY_IDS,
    COUNT(DISTINCT {{C_ENC}}) AS UNIQUE_ENCOUNTERS,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_COND}}) AS UNIQUE_CONDITIONS,
    COUNT(DISTINCT {{C_SNOMED}}) AS UNIQUE_SNOMED_CODES,
    COUNT(DISTINCT {{C_MEMBER}}) AS UNIQUE_FAMILY_MEMBERS,
    COUNT(DISTINCT {{C_STATUS}}) AS UNIQUE_STATUS_VALUES,
    COUNT(*) - COUNT(DISTINCT {{C_ID}}) AS EXTRA_ROWS_VS_UNIQUE_ID,
    ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT {{C_PT}}), 0), 2) AS AVG_ENTRIES_PER_PATIENT
FROM {{T}};

## 6. Completeness (nulls)

Required per the dictionary: id, encounter, patient, condition, status, family member, date. `SNOMED` is optional.

In [ ]:
SELECT
    COUNT(*) AS ROW_COUNT,
    SUM(IFF({{C_ID}} IS NULL, 1, 0)) AS NULL_FAMILY_HISTORY_ID,
    SUM(IFF({{C_ENC}} IS NULL, 1, 0)) AS NULL_ENCOUNTER_ID,
    SUM(IFF({{C_PT}} IS NULL, 1, 0)) AS NULL_PATIENT_ID,
    SUM(IFF({{C_SNOMED}} IS NULL, 1, 0)) AS NULL_SNOMED,
    SUM(IFF({{C_COND}} IS NULL, 1, 0)) AS NULL_CONDITION,
    SUM(IFF({{C_STATUS}} IS NULL, 1, 0)) AS NULL_STATUS,
    SUM(IFF({{C_MEMBER}} IS NULL, 1, 0)) AS NULL_FAMILY_MEMBER,
    SUM(IFF({{C_DATE}} IS NULL, 1, 0)) AS NULL_DATE
FROM {{T}};

## 7. Condition — unique values with occurrences

`Condition` is free text (for example `malignant neoplasm of lung`, `family history of cancer`). We group by the stored string; nothing is hardcoded.

`condition_counts` is the full list: one row per unique condition, `ROW_COUNT` = occurrences, plus the SNOMED codes attached to that name.

In [ ]:
SELECT
    COUNT(DISTINCT {{C_COND}}) AS UNIQUE_CONDITIONS,
    COUNT(DISTINCT UPPER(REGEXP_REPLACE(TRIM({{C_COND}}::STRING), '\\s+', ' '))) AS UNIQUE_CONDITIONS_NORMALIZED,
    SUM(IFF({{C_COND}} IS NULL, 1, 0)) AS NULL_CONDITION_ROWS,
    SUM(IFF(TRIM({{C_COND}}::STRING) = '', 1, 0)) AS BLANK_CONDITION_ROWS
FROM {{T}};

In [ ]:
SELECT
    {{C_COND}} AS CONDITION_NAME,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_ENC}}) AS UNIQUE_ENCOUNTERS,
    COUNT(DISTINCT {{C_MEMBER}}) AS DISTINCT_FAMILY_MEMBERS,
    COUNT(DISTINCT {{C_SNOMED}}) AS DISTINCT_SNOMED_CODES,
    LISTAGG(DISTINCT {{C_SNOMED}}::STRING, ' | ') AS SNOMED_CODES,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM {{T}}
GROUP BY 1
ORDER BY ROW_COUNT DESC, CONDITION_NAME;

### Word-level counts inside Condition

`malignant neoplasm of lung` is one condition name, but the word `neoplasm` also appears in other names. This splits `Condition` on spaces and punctuation and counts each word.

`WORD_OCCURRENCES` = how many rows contain that word.

In [ ]:
WITH tokens AS (
    SELECT
        {{C_PT}} AS PATIENT_ID,
        {{C_ENC}} AS ENCOUNTER_ID,
        TRIM(f.VALUE::STRING) AS WORD
    FROM {{T}},
         LATERAL FLATTEN(
             INPUT => SPLIT(
                 TRIM(REGEXP_REPLACE({{C_COND}}::STRING, '[^A-Za-z0-9]+', ' ')),
                 ' '
             )
         ) f
    WHERE {{C_COND}} IS NOT NULL
)
SELECT
    WORD,
    COUNT(*) AS WORD_OCCURRENCES,
    COUNT(DISTINCT PATIENT_ID) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT ENCOUNTER_ID) AS UNIQUE_ENCOUNTERS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_WORD_OCCURRENCES
FROM tokens
WHERE WORD IS NOT NULL
  AND WORD <> ''
GROUP BY 1
ORDER BY WORD_OCCURRENCES DESC, WORD;

## 8. SNOMED — codes, condition names, and same code with different names

Four views:

1. `snomed_unique_counts` — how many unique codes exist and how many rows carry one.
2. `snomed_condition_pairs` — **one row per code + condition name** with occurrences. `NAMES_FOR_THIS_CODE` tells you how many names that code has (4 means four different names) and `NAME_NUMBER` counts them 1, 2, 3, 4.
3. `snomed_name_variations` — one row per code, with all its names listed in a single column.
4. `snomed_same_code_many_names` — only the codes that have **more than one** name. This is the cleanup list.

`UNIQUE_NAMES_NORMALIZED` counts names after trimming, upper-casing, and collapsing spaces, so you can separate real naming differences from formatting noise.

In [ ]:
SELECT
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT NULLIF(TRIM({{C_SNOMED}}::STRING), '')) AS UNIQUE_SNOMED_CODES,
    SUM(IFF({{C_SNOMED}} IS NOT NULL AND TRIM({{C_SNOMED}}::STRING) <> '', 1, 0)) AS ROWS_WITH_SNOMED,
    SUM(IFF({{C_SNOMED}} IS NULL OR TRIM({{C_SNOMED}}::STRING) = '', 1, 0)) AS ROWS_MISSING_SNOMED,
    ROUND(
        100.0 * SUM(IFF({{C_SNOMED}} IS NOT NULL AND TRIM({{C_SNOMED}}::STRING) <> '', 1, 0))
        / NULLIF(COUNT(*), 0),
        2
    ) AS PCT_ROWS_WITH_SNOMED
FROM {{T}};

In [ ]:
SELECT
    {{C_SNOMED}} AS SNOMED,
    {{C_COND}} AS CONDITION_NAME,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(*) OVER (PARTITION BY {{C_SNOMED}}) AS NAMES_FOR_THIS_CODE,
    ROW_NUMBER() OVER (PARTITION BY {{C_SNOMED}} ORDER BY COUNT(*) DESC) AS NAME_NUMBER,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY {{C_SNOMED}}), 2) AS PCT_WITHIN_CODE
FROM {{T}}
WHERE {{C_SNOMED}} IS NOT NULL
  AND TRIM({{C_SNOMED}}::STRING) <> ''
GROUP BY 1, 2
ORDER BY NAMES_FOR_THIS_CODE DESC, SNOMED, ROW_COUNT DESC;

In [ ]:
SELECT
    {{C_SNOMED}} AS SNOMED,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_COND}}) AS UNIQUE_NAMES,
    COUNT(DISTINCT UPPER(REGEXP_REPLACE(TRIM({{C_COND}}::STRING), '\\s+', ' '))) AS UNIQUE_NAMES_NORMALIZED,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    LISTAGG(DISTINCT {{C_COND}}::STRING, ' | ') AS NAME_VARIATIONS
FROM {{T}}
WHERE {{C_SNOMED}} IS NOT NULL
  AND TRIM({{C_SNOMED}}::STRING) <> ''
GROUP BY 1
ORDER BY UNIQUE_NAMES DESC, ROW_COUNT DESC;

In [ ]:
SELECT
    {{C_SNOMED}} AS SNOMED,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_COND}}) AS UNIQUE_NAMES,
    LISTAGG(DISTINCT {{C_COND}}::STRING, ' | ') AS NAME_VARIATIONS
FROM {{T}}
WHERE {{C_SNOMED}} IS NOT NULL
  AND TRIM({{C_SNOMED}}::STRING) <> ''
GROUP BY 1
HAVING COUNT(DISTINCT {{C_COND}}) > 1
ORDER BY UNIQUE_NAMES DESC, ROW_COUNT DESC;

In [ ]:
SELECT
    {{C_COND}} AS CONDITION_NAME,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_SNOMED}}) AS UNIQUE_SNOMED_CODES,
    LISTAGG(DISTINCT {{C_SNOMED}}::STRING, ' | ') AS SNOMED_CODES
FROM {{T}}
WHERE {{C_COND}} IS NOT NULL
GROUP BY 1
HAVING COUNT(DISTINCT {{C_SNOMED}}) > 1
ORDER BY UNIQUE_SNOMED_CODES DESC, ROW_COUNT DESC;

## 9. FamilyMember and Status

`FamilyMember` says which relative the condition belongs to (Mother, Father, Sister, ...). Note the dictionary itself contains typos (`Borhter`, `Daugher`, `Grandaughter`), so expect spelling variants in the data — the counts below will expose them.

In [ ]:
SELECT
    {{C_MEMBER}} AS FAMILY_MEMBER,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_COND}}) AS UNIQUE_CONDITIONS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM {{T}}
GROUP BY 1
ORDER BY ROW_COUNT DESC;

In [ ]:
SELECT
    {{C_STATUS}} AS STATUS,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_COND}}) AS UNIQUE_CONDITIONS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM {{T}}
GROUP BY 1
ORDER BY ROW_COUNT DESC;

In [ ]:
SELECT
    {{C_MEMBER}} AS FAMILY_MEMBER,
    {{C_COND}} AS CONDITION_NAME,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM {{T}}
GROUP BY 1, 2
ORDER BY ROW_COUNT DESC;

In [ ]:
SELECT
    {{C_MEMBER}} AS FAMILY_MEMBER,
    {{C_STATUS}} AS STATUS,
    COUNT(*) AS ROW_COUNT,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM {{T}}
GROUP BY 1, 2
ORDER BY ROW_COUNT DESC;

## 10. Date distribution

`Date` is when the family history was **recorded**. Recency is today minus that date, so values stay positive; anything after today is flagged as future.

In [ ]:
SELECT
    CURRENT_DATE() AS TODAY,
    MIN({{C_DATE}}) AS MIN_DATE,
    MAX({{C_DATE}}) AS MAX_DATE,
    DATEDIFF('day', MIN({{C_DATE}})::DATE, MAX({{C_DATE}})::DATE) AS SPAN_DAYS,
    SUM(IFF({{C_DATE}}::DATE > CURRENT_DATE(), 1, 0)) AS FUTURE_ROWS,
    SUM(IFF({{C_DATE}} IS NULL, 1, 0)) AS MISSING_DATE_ROWS
FROM {{T}};

In [ ]:
SELECT
    YEAR({{C_DATE}}) AS RECORD_YEAR,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_COND}}) AS UNIQUE_CONDITIONS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM {{T}}
GROUP BY 1
ORDER BY 1;

In [ ]:
WITH bucketed AS (
    SELECT
        CASE
            WHEN {{C_DATE}} IS NULL THEN 90
            WHEN {{C_DATE}}::DATE > CURRENT_DATE() THEN 80
            WHEN DATEDIFF('day', {{C_DATE}}::DATE, CURRENT_DATE()) <= 30 THEN 1
            WHEN DATEDIFF('day', {{C_DATE}}::DATE, CURRENT_DATE()) <= 90 THEN 2
            WHEN DATEDIFF('day', {{C_DATE}}::DATE, CURRENT_DATE()) <= 180 THEN 3
            WHEN DATEDIFF('day', {{C_DATE}}::DATE, CURRENT_DATE()) <= 365 THEN 4
            WHEN DATEDIFF('year', {{C_DATE}}::DATE, CURRENT_DATE()) <= 2 THEN 5
            WHEN DATEDIFF('year', {{C_DATE}}::DATE, CURRENT_DATE()) <= 5 THEN 6
            WHEN DATEDIFF('year', {{C_DATE}}::DATE, CURRENT_DATE()) <= 7 THEN 7
            WHEN DATEDIFF('year', {{C_DATE}}::DATE, CURRENT_DATE()) <= 10 THEN 8
            ELSE 9
        END AS SORT_ORDER,
        {{C_PT}} AS PATIENT_ID
    FROM {{T}}
)
SELECT
    SORT_ORDER,
    CASE SORT_ORDER
        WHEN 1 THEN '0-30 days ago'
        WHEN 2 THEN '31-90 days ago'
        WHEN 3 THEN '91-180 days ago'
        WHEN 4 THEN '181-365 days ago'
        WHEN 5 THEN '1-2 years ago'
        WHEN 6 THEN '3-5 years ago'
        WHEN 7 THEN '6-7 years ago'
        WHEN 8 THEN '8-10 years ago'
        WHEN 9 THEN 'More than 10 years ago'
        WHEN 80 THEN 'Future (after today)'
        ELSE 'Unknown (missing date)'
    END AS DATE_RANGE,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT PATIENT_ID) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM bucketed
GROUP BY 1, 2
ORDER BY SORT_ORDER;

## 11. Entries per patient, and one patient drill-down

The drill-down auto-picks the patient with the most rows. To inspect a specific patient, replace the subquery in the `WHERE` clause with that id.

In [ ]:
WITH per_patient AS (
    SELECT
        {{C_PT}} AS PATIENT_ID,
        COUNT(*) AS ROW_COUNT
    FROM {{T}}
    WHERE {{C_PT}} IS NOT NULL
    GROUP BY 1
)
SELECT
    ROW_COUNT AS ENTRIES_PER_PATIENT,
    COUNT(*) AS NUMBER_OF_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_PATIENTS
FROM per_patient
GROUP BY 1
ORDER BY 1;

In [ ]:
SELECT
    {{C_PT}} AS PATIENT_ID,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_COND}}) AS UNIQUE_CONDITIONS,
    COUNT(DISTINCT {{C_MEMBER}}) AS UNIQUE_FAMILY_MEMBERS,
    COUNT(DISTINCT {{C_SNOMED}}) AS UNIQUE_SNOMED_CODES,
    MIN({{C_DATE}}) AS FIRST_DATE,
    MAX({{C_DATE}}) AS LAST_DATE
FROM {{T}}
WHERE {{C_PT}} IS NOT NULL
GROUP BY 1
ORDER BY ROW_COUNT DESC
LIMIT 25;

In [ ]:
SELECT
    {{C_PT}} AS PATIENT_ID,
    {{C_ID}} AS FAMILY_HISTORY_ID,
    {{C_ENC}} AS ENCOUNTER_ID,
    {{C_MEMBER}} AS FAMILY_MEMBER,
    {{C_COND}} AS CONDITION_NAME,
    {{C_SNOMED}} AS SNOMED,
    {{C_STATUS}} AS STATUS,
    {{C_DATE}} AS RECORD_DATE
FROM {{T}}
WHERE {{C_PT}} = (
        SELECT {{C_PT}}
        FROM {{T}}
        WHERE {{C_PT}} IS NOT NULL
        GROUP BY 1
        ORDER BY COUNT(*) DESC
        LIMIT 1
      )
ORDER BY {{C_DATE}} NULLS LAST, {{C_ID}};

## Notes

- **Downloading:** run a SQL cell, then use the download arrow on that result grid.
- If a cell fails with **invalid identifier**, copy names from `describe_table` into `COL`. `EncounterId/VisitId` and `Member/PatientId` need `QUOTE_COLUMNS = True`.
- `LISTAGG` caps at 16 MB per group. If a name-variation cell errors because one code has a huge number of long names, use `snomed_condition_pairs` instead — same information, one row per pair.
- Run `config` before the SQL cells.